# Try cuplyr on a free Colab GPU

[cuplyr](https://github.com/bbtheo/cuplyr) runs dplyr code on an NVIDIA GPU through RAPIDS libcudf.
This notebook installs it on Colab and runs a few pipelines.

1. **Runtime → Change runtime type**, choose **T4 GPU** and keep the runtime type as **Python 3**.
2. **Runtime → Run all.**

The first cell installs everything, which takes a few minutes. It runs once per Colab session.

In [ ]:
#@title Install cuplyr (a few minutes) { display-mode: "form" }
# Installs R, RAPIDS libcudf and the cuplyr conda package into /opt/cuplyr, then adds a %%R cell magic
# that sends each cell to one R session kept open for the whole notebook.
import os, subprocess, tempfile, uuid
from IPython.core.magic import register_cell_magic

PREFIX = os.environ.get("CUPLYR_PREFIX", "/opt/cuplyr")
R_BIN = f"{PREFIX}/bin/R"
LOG = "/tmp/cuplyr-install.log"

INSTALL_SH = r'''#!/usr/bin/env bash
# R, RAPIDS libcudf and the prebuilt cuplyr package all come from one micromamba env,
# so nothing mixes with the libraries of the host's own R or Python.
set -euo pipefail
PREFIX="${PREFIX:-/opt/cuplyr}"
MAMBA_ROOT="${MAMBA_ROOT:-/opt/micromamba}"
LOG="${LOG:-/tmp/cuplyr-install.log}"
: > "$LOG"

echo "==> GPU: $(nvidia-smi --query-gpu=name,memory.total --format=csv,noheader)"

# The r-cuplyr package is built against libcudf 26.10 for CUDA 13, so the driver has to support CUDA 13.
drv=$(nvidia-smi | grep -oE 'CUDA (UMD )?Version: [0-9]+\.[0-9]+' | grep -oE '[0-9]+\.[0-9]+' | head -1)
if [ -z "$drv" ] || [ "${drv%%.*}" -lt 13 ]; then
  echo "ERROR: this runtime's NVIDIA driver supports CUDA ${drv:-(unknown)}, and cuplyr needs CUDA 13." >&2
  exit 1
fi

mm="$MAMBA_ROOT/bin/micromamba"
if [ ! -x "$mm" ]; then
  mkdir -p "$MAMBA_ROOT/bin"
  curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C "$MAMBA_ROOT" bin/micromamba
fi
export MAMBA_ROOT_PREFIX="$MAMBA_ROOT"

echo "==> Downloading R, RAPIDS libcudf 26.10 and cuplyr from the bbtheo conda channel"
"$mm" create -y -p "$PREFIX" -c bbtheo -c rapidsai -c conda-forge -c nvidia \
    r-cuplyr \
    >> "$LOG" 2>&1
echo "==> Installed cuplyr $("$PREFIX/bin/Rscript" -e 'cat(format(packageVersion("cuplyr")))')"
'''

if not os.path.exists(f"{PREFIX}/lib/R/library/cuplyr"):
    with open("/tmp/install-cuplyr.sh", "w") as f:
        f.write(INSTALL_SH)
    env = dict(os.environ, PREFIX=PREFIX, LOG=LOG)
    proc = subprocess.Popen(["bash", "/tmp/install-cuplyr.sh"], env=env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        print(open(LOG).read()[-5000:])
        raise RuntimeError(f"Install failed; the full log is in {LOG}")
else:
    print("cuplyr is already installed")

_SENTINEL = "__cell_done_" + uuid.uuid4().hex
_r = None

def _start_r():
    global _r
    _r = subprocess.Popen([R_BIN, "--vanilla", "--quiet", "--no-echo"], text=True, bufsize=1,
                          stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    _r.stdin.write("options(warn = 1, width = 100); sink(stdout(), type = 'message')\n"
                   ".run_cell <- function(f) tryCatch(\n"
                   "  source(f, local = globalenv(), print.eval = TRUE, max.deparse.length = Inf),\n"
                   "  error = function(e) cat('Error: ', conditionMessage(e), '\\n', sep = ''))\n")
    _r.stdin.flush()

def _run(code):
    with tempfile.NamedTemporaryFile("w", suffix=".R", delete=False) as f:
        f.write(code)
    _r.stdin.write(f".run_cell('{f.name}'); cat('\\n{_SENTINEL}\\n'); flush(stdout())\n")
    _r.stdin.flush()
    for line in _r.stdout:
        if line.rstrip("\n") == _SENTINEL:
            break
        print(line, end="", flush=True)
    else:
        raise RuntimeError("R exited; run the cell again to start a new R session")
    os.unlink(f.name)

@register_cell_magic
def R(line, cell):
    if _r is None or _r.poll() is not None:
        _start_r()
    _run(cell)

print("Ready: start a cell with %%R to run R")


Every cell below starts with `%%R`. That line sends the cell to an R session that stays open,
so variables carry over from one cell to the next. Everything else is plain R.
Loading cuplyr prints the GPU it found.

In [ ]:
%%R
library(dplyr, warn.conflicts = FALSE)
library(cuplyr, warn.conflicts = FALSE)

## A first pipeline

`tbl_gpu()` copies a data frame to the GPU. The dplyr verbs then run there, and `collect()` brings the result back as a tibble.

In [ ]:
%%R
set.seed(1)
n <- 1e5
sales <- data.frame(
  year    = sample(2018:2024, n, replace = TRUE),
  region  = sample(c("north", "south", "east", "west"), n, replace = TRUE),
  quarter = sample(1:4, n, replace = TRUE),
  amount  = rnorm(n, mean = 50, sd = 30),
  price   = runif(n, 1, 10)
)
regions <- data.frame(
  region  = c("north", "south", "east", "west"),
  manager = c("Ana", "Ben", "Chi", "Dev")
)

tbl_gpu(sales) |>
  filter(year >= 2020, amount > 0) |>
  mutate(revenue = amount * price) |>
  group_by(region, quarter) |>
  summarise(total = sum(revenue), .groups = "drop") |>
  inner_join(tbl_gpu(regions), by = "region") |>
  arrange(desc(total)) |>
  collect()

## Lazy mode

With `lazy = TRUE` each verb only records a step. `collect()` optimizes the whole plan, runs it,
and frees the intermediates. `show_query()` prints the plan.

In [ ]:
%%R
lazy_sales <- tbl_gpu(sales, lazy = TRUE) |>
  filter(amount > 0) |>
  mutate(revenue = amount * price) |>
  group_by(region) |>
  summarise(total = sum(revenue), mean_price = mean(price))

show_query(lazy_sales)
collect(lazy_sales)

## Is it faster?

The same summary on 2e7 rows with dplyr and with cuplyr. The data is copied to the GPU once,
outside the timing, and each timing is the median of five runs after one warm-up run.
A T4 is a 2018 card and Colab gives you two CPU cores, so expect different ratios from
the [benchmarks](https://github.com/bbtheo/cuplyr-material), which used an RTX 5070 and 24 CPU threads.

In [ ]:
%%R
set.seed(2)
n <- 2e7
big <- data.frame(
  key   = sample(1e4, n, replace = TRUE),
  x     = runif(n),
  y     = rnorm(n)
)
big_gpu <- tbl_gpu(big)

pipeline <- function(d) d |>
  filter(x > 0.2) |>
  mutate(z = x * y) |>
  group_by(key) |>
  summarise(total = sum(z), n = n(), .groups = "drop") |>
  collect()

median_ms <- function(f, reps = 5) {
  f()
  median(replicate(reps, system.time(f())[["elapsed"]])) * 1000
}

t_dplyr  <- median_ms(function() pipeline(big))
t_cuplyr <- median_ms(function() pipeline(big_gpu))
cat(sprintf("dplyr:  %7.0f ms\ncuplyr: %7.0f ms\nratio:  %7.1fx\n", t_dplyr, t_cuplyr, t_dplyr / t_cuplyr))

all.equal(
  arrange(pipeline(big), key),
  arrange(pipeline(big_gpu), key),
  check.attributes = FALSE
)

## Your turn

Edit this cell and run it. Supported verbs: `filter()`, `select()`, `mutate()`, `arrange()`,
`group_by()` with `summarise()`, and the four mutating joins. If something breaks, please
[open an issue](https://github.com/bbtheo/cuplyr/issues).

In [ ]:
%%R
tbl_gpu(mtcars) |>
  group_by(cyl) |>
  summarise(n = n(), mpg = mean(mpg), hp = mean(hp)) |>
  arrange(cyl) |>
  collect()